In [2]:
import pandas as pd
import polars as pl

In [53]:
# Dicionário contendo os nomes das tabelas e arquivos CSV
tabelas = {
    'estabelecimento': 'tbEstabelecimento202409.csv',
    'tipo_estabelecimento': 'tbTipoEstabelecimento202409.csv',
    'atividade_principal': 'tbAtividade202409.csv',
    'natureza_juridica': 'tbNaturezaJuridica202409.csv',
    'motivo_desativacao': 'tbMotivoDesativacao202409.csv',
    'cod_municipio': 'tbMunicipio202409.csv'
}

# Caminho base onde estão localizados os arquivos CSV
file_path = "certeza_de_uso/BASE_DE_DADOS_CNES_202409/"

# Dicionário para armazenar os dataframes
dataframes = {}

# Loop para carregar cada CSV e armazenar no dicionário
for var, tabela in tabelas.items():
    try:
        # Carregar o CSV usando Polars
        df = pl.read_csv(
            file_path + tabela,
            separator=';',          # Delimitador do CSV
            encoding='utf8-lossy',  # Codificação com fallback para caracteres problemáticos
            has_header=True,        # Assume que a primeira linha é o cabeçalho
            ignore_errors=True      # Ignora possíveis erros de parsing
        )
        # Armazena o dataframe no dicionário usando a chave do loop
        dataframes[var] = df
        print(f"Carregado: {var} com {df.shape[0]} linhas e {df.shape[1]} colunas")
    
    except Exception as e:
        print(f"Erro ao carregar o arquivo {tabela}: {e}")

print(f"Tamanho do datraframe de estabelecimentos: {dataframes['estabelecimento'].shape}")

dataframes['tipo_estabelecimento'] = dataframes['tipo_estabelecimento'].with_columns(
    pl.col('CO_TIPO_ESTABELECIMENTO').cast(pl.Utf8)
)

dataframes['motivo_desativacao'] = dataframes['motivo_desativacao'].with_columns(
    pl.col('CD_MOTIVO_DESAB').cast(pl.Utf8)
)

# Realizar os joins de forma incremental usando 'left' join
df = dataframes['estabelecimento']

df = df.join(
    dataframes['tipo_estabelecimento'], 
    on='CO_TIPO_ESTABELECIMENTO', 
    how='left'
)

df = df.join(
    dataframes['atividade_principal'], 
    on='CO_ATIVIDADE', 
    how='left'
)

df = df.join(
    dataframes['natureza_juridica'], 
    on='CO_NATUREZA_JUR', 
    how='left'
)

df = df.join(
    dataframes['motivo_desativacao'], 
    left_on='CO_MOTIVO_DESAB', 
    right_on='CD_MOTIVO_DESAB', 
    how='left'
)

df = df.join(
    dataframes['cod_municipio'], 
    left_on='CO_MUNICIPIO_GESTOR', 
    right_on='CO_MUNICIPIO', 
    how='left'
)

print(f"Dataframe final após os joins: {df.shape}")

Carregado: estabelecimento com 548991 linhas e 56 colunas
Carregado: tipo_estabelecimento com 26 linhas e 3 colunas
Carregado: atividade_principal com 28 linhas e 4 colunas
Carregado: natureza_juridica com 100 linhas e 2 colunas
Carregado: motivo_desativacao com 14 linhas e 2 colunas
Carregado: cod_municipio com 5606 linhas e 13 colunas
Tamanho do datraframe de estabelecimentos: (548991, 56)
Dataframe final após os joins: (548991, 75)


In [65]:
# Análise das tabelas carregadas (exceto 'estabelecimento')
for tabela, df in dataframes.items():
    if tabela != 'estabelecimento':
        print(f"\n-------------------------- {tabela} ------------------------------------------------")
        # Exibir contagem de valores por coluna
        for col in df.columns:
            print(f"Coluna: {col}")
            print(df[col].value_counts(), "\n")


-------------------------- tipo_estabelecimento ------------------------------------------------
Coluna: CO_TIPO_ESTABELECIMENTO
shape: (26, 2)
┌─────────────────────────┬───────┐
│ CO_TIPO_ESTABELECIMENTO ┆ count │
│ ---                     ┆ ---   │
│ str                     ┆ u32   │
╞═════════════════════════╪═══════╡
│ 15                      ┆ 1     │
│ 4                       ┆ 1     │
│ 12                      ┆ 1     │
│ 18                      ┆ 1     │
│ 3                       ┆ 1     │
│ …                       ┆ …     │
│ 24                      ┆ 1     │
│ 14                      ┆ 1     │
│ 11                      ┆ 1     │
│ 23                      ┆ 1     │
│ 8                       ┆ 1     │
└─────────────────────────┴───────┘ 

Coluna: DS_TIPO_ESTABELECIMENTO
shape: (26, 2)
┌─────────────────────────────────┬───────┐
│ DS_TIPO_ESTABELECIMENTO         ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════

In [56]:
df.filter(
    pl.col('NO_MUNICIPIO') == 'PARANAGUA'
).head()

CO_UNIDADE,CO_CNES,NU_CNPJ_MANTENEDORA,TP_PFPJ,NIVEL_DEP,NO_RAZAO_SOCIAL,NO_FANTASIA,NO_LOGRADOURO,NU_ENDERECO,NO_COMPLEMENTO,NO_BAIRRO,CO_CEP,CO_REGIAO_SAUDE,CO_MICRO_REGIAO,CO_DISTRITO_SANITARIO,CO_DISTRITO_ADMINISTRATIVO,NU_TELEFONE,NU_FAX,NO_EMAIL,NU_CPF,NU_CNPJ,CO_ATIVIDADE,CO_CLIENTELA,NU_ALVARA,DT_EXPEDICAO,TP_ORGAO_EXPEDIDOR,DT_VAL_LIC_SANI,TP_LIC_SANI,TP_UNIDADE,CO_TURNO_ATENDIMENTO,CO_ESTADO_GESTOR,CO_MUNICIPIO_GESTOR,"TO_CHAR(DT_ATUALIZACAO,'DD/MM/YYYY')",CO_USUARIO,CO_CPFDIRETORCLN,REG_DIRETORCLN,ST_ADESAO_FILANTROP,CO_MOTIVO_DESAB,NO_URL,NU_LATITUDE,NU_LONGITUDE,"TO_CHAR(DT_ATU_GEO,'DD/MM/YYYY')",NO_USUARIO_GEO,CO_NATUREZA_JUR,TP_ESTAB_SEMPRE_ABERTO,ST_GERACREDITO_GERENTE_SGIF,ST_CONEXAO_INTERNET,CO_TIPO_UNIDADE,NO_FANTASIA_ABREV,TP_GESTAO,"TO_CHAR(DT_ATUALIZACAO_ORIGEM,'DD/MM/YYYY')",CO_TIPO_ESTABELECIMENTO,CO_ATIVIDADE_PRINCIPAL,ST_CONTRATO_FORMALIZADO,CO_TIPO_ABRANGENCIA,ST_COWORKING,DS_TIPO_ESTABELECIMENTO,DS_CONCEITO_TIPO,CO_GRUPO_ATIVIDADE,DS_ATIVIDADE,DS_CONCEITO_ATIVIDADE,DS_NATUREZA_JUR,DS_MOTIVO_DESAB,NO_MUNICIPIO,CO_SIGLA_ESTADO,TP_CADASTRO,TP_PACTO,TP_ENVIA,TP_ENVIA_CNES,TP_CIB_SAS,TP_PLENO_ORIGEM,TP_MAC,NU_POPULACAO,NU_DENSIDADE,CMTP_INICIO_MAC
i64,i64,str,i64,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,i64,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,i64
4118209553088,9553088,"""""",3,1,"""LABAN LABORATORIO DE ANALISES …","""LABAN""","""GABRIEL DE LARA""","""658""","""""","""29 DE JULHO""",83221685,"""1�""","""""","""""","""""","""41 30336868""","""""","""financeiro@laban.com.br""","""""","""72082043001731""",4,1,"""37/2018""","""22-mai-2018 00:00:00""","""2 ""","""22-mai-2019 00:00:00""","""1""",39,3,41,411820,"""10/02/2022""","""SEMSA""","""00972756973""","""24329""","""""","""""","""www.labanlaboratorio.com.br""","""-25.52""","""-48.509""","""21/08/2020""","""SEMSA""",2062,"""N""","""""","""S""","""""","""""","""D""","""19/08/2018""","""018""","""002""","""N""","""""","""""",null,null,1,"""REABILITACAO""","""Conjunto de a��es e servi�os o…","""SOCIEDADE EMPRESARIA LIMITADA""",null,"""PARANAGUA""","""PR""","""S""","""S""",1,"""S""","""S""","""N""","""S""","""""","""1""",32018
4118209814302,9814302,"""""",3,1,"""ALMEIDA E MODELLI EMPREENDIMEN…","""ALMEIDA E MODELLI EMPREENDIMEN…","""FLORINDA CARLOS CARDOSO""","""667""","""CASA 1""","""DIVINEIA""",83212420,"""01""","""""","""""","""""","""31 37736509""","""""","""nitida@nitidacontabil.com.br""","""""","""30547757000160""",4,3,"""139/2018""","""24-jul-2018 00:00:00""","""2 ""","""24-jul-2019 00:00:00""","""2""",22,3,41,411820,"""05/07/2022""","""SEMSA""","""03435410922""","""08/23117""","""""","""""","""""","""-25.52""","""-48.509""","""05/06/2019""","""SEMSA""",2062,"""N""","""""","""S""","""""","""""","""M""","""25/06/2019""","""016""","""001""","""""","""""","""""",null,null,1,"""REABILITACAO""","""Conjunto de a��es e servi�os o…","""SOCIEDADE EMPRESARIA LIMITADA""",null,"""PARANAGUA""","""PR""","""S""","""S""",1,"""S""","""S""","""N""","""S""","""""","""1""",32018
4118209920145,9920145,"""76017458000115""",3,3,"""MUNICIPIO DE PARANAGUA""","""UNIDADE DE SAUDE MARIA VARGAS …","""ANTONIO CARLOS RODRIGUES""","""S/N""","""""","""JARDIM OURO FINO""",83215750,"""01""","""""","""""","""""","""41-34202748""","""""","""""","""""","""""",4,3,"""1453/2019""","""06-set-2019 00:00:00""","""2 ""","""06-set-2020 00:00:00""","""2""",2,3,41,411820,"""10/06/2024""","""SEMSA""","""11740646762""","""466374""","""""","""""","""""","""-25.52""","""-48.509""","""20/01/2023""","""SEMSA""",1244,"""N""","""""","""S""","""""","""""","""M""","""25/09/2019""","""001""","""012""","""""","""""","""""",null,null,1,"""REABILITACAO""","""Conjunto de a��es e servi�os o…","""MUNICIPIO""",null,"""PARANAGUA""","""PR""","""S""","""S""",1,"""S""","""S""","""N""","""S""","""""","""1""",32018
4118200180351

In [61]:
df['CO_SIGLA_ESTADO'].value_counts().sort('count')

CO_SIGLA_ESTADO,count
str,u32
"""AP""",1169
"""RR""",1461
"""AC""",1760
"""TO""",3110
"""AM""",4029
…,…
"""PR""",38763
"""RS""",42176
"""RJ""",49962


In [64]:
df['NO_MUNICIPIO'].value_counts().sort('count', descending=True).head(20)

NO_MUNICIPIO,count
str,u32
"""SAO PAULO""",37631
"""RIO DE JANEIRO""",24878
"""BRASILIA""",10923
"""BELO HORIZONTE""",9682
"""CURITIBA""",9242
…,…
"""BELEM""",2949
"""JOAO PESSOA""",2814
"""FLORIANOPOLIS""",2769


In [66]:
df.group_by(
    pl.col('NO_MUNICIPIO')
).agg(
    pl.col('NU_POPULACAO')
)

NO_MUNICIPIO,NU_POPULACAO
str,list[str]
"""ALVARES MACHADO""","[""""]"
"""BACURITUBA""","[""""]"
"""GUAJARA-MIRIM""","[""42694""]"
"""PRESIDENTE ALVES""","[""""]"
"""JANAUBA""","[""""]"
…,…
"""ITAPEVA""","["""", """"]"
"""SAO JOSE DO POVO""","[""""]"
"""LARANJAL DO JARI""","[""37626""]"


In [35]:
dataframes['motivo_desativacao'].with_columns(
    pl.col('CD_MOTIVO_DESAB').cast(pl.Utf8)
)

CD_MOTIVO_DESAB,DS_MOTIVO_DESAB
str,str
"""1""","""DESATIVADO TEMPORARIAMENTE PEL…"
"""3""","""DESATIVADO TEMPORARIAMENTE POR…"
"""2""","""DESATIVADO TEMPORARIAMENTE POR…"
"""4""","""DESATIVADO - OUTROS"""
"""6""","""PELO GESTOR POR DESATUALIZACAO…"
…,…
"""12""","""ALTERACAO DO REGIME DE DIREITO…"
"""14""","""PELO GESTOR POR INCONSISTENCIA…"
"""10""","""ENCERRAMENTO DAS ATIVIDADES"""


In [40]:
dataframes['estabelecimento'].join(dataframes['cod_municipio'], left_on='CO_MUNICIPIO_GESTOR', right_on='CO_MUNICIPIO')

CO_UNIDADE,CO_CNES,NU_CNPJ_MANTENEDORA,TP_PFPJ,NIVEL_DEP,NO_RAZAO_SOCIAL,NO_FANTASIA,NO_LOGRADOURO,NU_ENDERECO,NO_COMPLEMENTO,NO_BAIRRO,CO_CEP,CO_REGIAO_SAUDE,CO_MICRO_REGIAO,CO_DISTRITO_SANITARIO,CO_DISTRITO_ADMINISTRATIVO,NU_TELEFONE,NU_FAX,NO_EMAIL,NU_CPF,NU_CNPJ,CO_ATIVIDADE,CO_CLIENTELA,NU_ALVARA,DT_EXPEDICAO,TP_ORGAO_EXPEDIDOR,DT_VAL_LIC_SANI,TP_LIC_SANI,TP_UNIDADE,CO_TURNO_ATENDIMENTO,CO_ESTADO_GESTOR,CO_MUNICIPIO_GESTOR,"TO_CHAR(DT_ATUALIZACAO,'DD/MM/YYYY')",CO_USUARIO,CO_CPFDIRETORCLN,REG_DIRETORCLN,ST_ADESAO_FILANTROP,CO_MOTIVO_DESAB,NO_URL,NU_LATITUDE,NU_LONGITUDE,"TO_CHAR(DT_ATU_GEO,'DD/MM/YYYY')",NO_USUARIO_GEO,CO_NATUREZA_JUR,TP_ESTAB_SEMPRE_ABERTO,ST_GERACREDITO_GERENTE_SGIF,ST_CONEXAO_INTERNET,CO_TIPO_UNIDADE,NO_FANTASIA_ABREV,TP_GESTAO,"TO_CHAR(DT_ATUALIZACAO_ORIGEM,'DD/MM/YYYY')",CO_TIPO_ESTABELECIMENTO,CO_ATIVIDADE_PRINCIPAL,ST_CONTRATO_FORMALIZADO,CO_TIPO_ABRANGENCIA,ST_COWORKING,NO_MUNICIPIO,CO_SIGLA_ESTADO,TP_CADASTRO,TP_PACTO,TP_ENVIA,TP_ENVIA_CNES,TP_CIB_SAS,TP_PLENO_ORIGEM,TP_MAC,NU_POPULACAO,NU_DENSIDADE,CMTP_INICIO_MAC
i64,i64,str,i64,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,i64,i64,str,str,str,str,str,i64,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,i64
2609602569302,2569302,"""10404184000109""",3,3,"""PREFEITURA MUNICIPAL DE OLINDA""","""USF BULTRINS MONTE II""","""RUA PREFEITO MANOEL REGUEIRA""","""540""","""""","""BULTRINS""",53320460,"""001""","""""","""""","""""","""(81)34930626""","""""","""""","""""","""""",4,2,"""""","""""","""""","""""","""""",2,3,26,260960,"""28/11/2017""","""ANA KARLA""","""10258000449""","""157002""","""""","""04""","""""","""""","""""","""""","""""",1244,"""N""","""""","""S""","""""","""""","""M""","""18/06/2003""","""""","""""","""""","""""","""""","""OLINDA""","""PE""","""S""","""S""",1,"""S""","""S""","""S""","""S""","""""","""9""",32018
2609602571943,2571943,"""10404184000109""",3,3,"""PREFEITURA MUNICIPAL DE OLINDA""","""UNIDADE MOVEL""","""RUA DO SOL""","""311""","""""","""CARMO""",53120010,"""001""","""""","""02""","""""","""(81)34294465""","""""","""""","""""","""""",4,1,"""""","""""","""""","""""","""""",40,1,26,260960,"""02/10/2024""","""ANA KARLA""","""68887302472""","""6191""","""""","""""","""""","""-8.0105168""","""-34.8427588""","""06/07/2023""","""ANA""",1244,"""N""","""""","""N""","""""","""""","""M""","""18/06/2003""","""016""","""001""","""""","""""","""N""","""OLINDA""","""PE""","""S""","""S""",1,"""S""","""S""","""S""","""S""","""""","""9""",32018
2609602344637,2344637,"""10404184000109""",3,3,"""PREFEITURA MUNICIPAL DE OLINDA""","""USF ALTO DA MINA""","""RUA AVENCA""","""49""","""""","""ALTO DA MINA""",53250441,"""001""","""""","""02""","""""","""(81)33051133""","""""","""""","""""","""""",4,1,"""""","""""","""""","""""","""""",2,3,26,260960,"""02/10/2024""","""ANA KARLA""","""03760998445""","""322481""","""""","""""","""""","""-7.9944275""","""-34.8534609""","""03/08/2022""","""ANA""",1244,"""N""","""""","""S""","""""","""""","""M""","""30/10/2001""","""001""","""012""","""""","""""","""N""","""OLINDA""","""PE""","""S""","""S""",1,"""S""","""S""","""S""","""S""","""""","""9""",32018
2609602344696,2344696,"""10404184000109""",3,3,"""PREFEITURA MUNICIPAL DE OLINDA""","""USF ILHA DE SANTANA I E II""","""RUA DA INTEGRACAO""","""S/N""","""""","""JARDIM ATLANTICO""",53060001,"""001""","""""","""02""","""""","""(81)34324703""","""""","""""","""""","""""",4,1,"""""","""""","""""","""""","""""",2,3,26,260960,"""02/10/2024""","""ANA KARLA""","""58346740468""","""1399172""","""""","""""","""""","""-8.0412002""","""-34.879982""","""06/07/2023""","""ANA""",1244,"""N""","""""","""S""","""""","""""","""M""","""30/10/2001""","""001""","""012""","""""","""""","""N""","""OLINDA""","""PE""","""S""","""S""",1,"""S""","""S""","""S""","""S""","""""","""9""",32018
3112002142295,2142295,"""""",3,1,"""FUNDACAO COMUNITARIA DE SAUDE …","""HOS

ComputeError: datatypes of join keys don't match - `CO_TIPO_ESTABELECIMENTO`: str on left does not match `CO_TIPO_ESTABELECIMENTO`: i64 on right

In [ ]:
tabelas = {
    'estabelecimento': 'tbEstabelecimento202409.csv',
    'tipo_estabelecimento': 'tbTipoEstabelecimento202409.csv',
    'atividade_principal': 'tbAtividade202409.csv',
    'natureza_juridica': 'tbNaturezaJuridica202409.csv',
    'motivo_desativacao': 'tbMotivoDesativacao202409.csv',
    'cod_municipio': 'tbMunicipio202409.csv'
}
file_path = "certeza_de_uso/BASE_DE_DADOS_CNES_202409/"


for var, tabela in tabelas

try:
    df = pl.read_csv(
        file_path,
        separator=';',                # Definir delimitador como ';'
        encoding='utf8-lossy',  # Tenta carregar com utf-8, ignorando caracteres problemáticos
        has_header=True,        # Assume que a primeira linha é o cabeçalho
        ignore_errors=True      # Ignora possíveis erros de parsing
    )
    print(df.head())
except Exception as e:
    print(f"Erro ao carregar o CSV: {e}")

shape: (5, 56)
┌────────────┬─────────┬────────────┬─────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ CO_UNIDADE ┆ CO_CNES ┆ NU_CNPJ_MA ┆ TP_PFPJ ┆ … ┆ CO_ATIVIDA ┆ ST_CONTRA ┆ CO_TIPO_A ┆ ST_COWORK │
│ ---        ┆ ---     ┆ NTENEDORA  ┆ ---     ┆   ┆ DE_PRINCIP ┆ TO_FORMAL ┆ BRANGENCI ┆ ING       │
│ i64        ┆ i64     ┆ ---        ┆ i64     ┆   ┆ AL         ┆ IZADO     ┆ A         ┆ ---       │
│            ┆         ┆ str        ┆         ┆   ┆ ---        ┆ ---       ┆ ---       ┆ str       │
│            ┆         ┆            ┆         ┆   ┆ str        ┆ str       ┆ str       ┆           │
╞════════════╪═════════╪════════════╪═════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 2609602569 ┆ 2569302 ┆ 1040418400 ┆ 3       ┆ … ┆            ┆           ┆           ┆           │
│ 302        ┆         ┆ 0109       ┆         ┆   ┆            ┆           ┆           ┆           │
│ 2609602571 ┆ 2571943 ┆ 1040418400 ┆ 3       ┆ … ┆ 001        ┆           ┆

In [24]:
for col in df.columns:
    print(df[col].value_counts(), '\n')

shape: (548_835, 2)
┌───────────────┬───────┐
│ CO_UNIDADE    ┆ count │
│ ---           ┆ ---   │
│ i64           ┆ u32   │
╞═══════════════╪═══════╡
│ 3543407007426 ┆ 1     │
│ 5103406584276 ┆ 1     │
│ 1504209970436 ┆ 1     │
│ 1709507343590 ┆ 1     │
│ 4310605257395 ┆ 1     │
│ …             ┆ …     │
│ 3503309834311 ┆ 1     │
│ 3550309602100 ┆ 1     │
│ 4209300890561 ┆ 1     │
│ 2210602884445 ┆ 1     │
│ 3554505051339 ┆ 1     │
└───────────────┴───────┘ 

shape: (548_991, 2)
┌─────────┬───────┐
│ CO_CNES ┆ count │
│ ---     ┆ ---   │
│ i64     ┆ u32   │
╞═════════╪═══════╡
│ 7616961 ┆ 1     │
│ 2286629 ┆ 1     │
│ 612561  ┆ 1     │
│ 5218152 ┆ 1     │
│ 6193501 ┆ 1     │
│ …       ┆ …     │
│ 2953757 ┆ 1     │
│ 7070012 ┆ 1     │
│ 4278518 ┆ 1     │
│ 3506908 ┆ 1     │
│ 6243045 ┆ 1     │
└─────────┴───────┘ 

shape: (7_264, 2)
┌─────────────────────┬───────┐
│ NU_CNPJ_MANTENEDORA ┆ count │
│ ---                 ┆ ---   │
│ str                 ┆ u32   │
╞═════════════════════╪═════

In [10]:
df.group_by(
    pl.col('CO_UNIDADE')
).agg(
    pl.col('NO_RAZAO_SOCIAL').n_unique(),
    pl.col('NO_FANTASIA').n_unique()
).filter(
    pl.col('NO_RAZAO_SOCIAL') > 1
)

CO_UNIDADE,NO_RAZAO_SOCIAL,NO_FANTASIA
i64,u32,u32
null,135,155
